In [ ]:
import pandas as pd

# escolher o número de componentes
# número de componentes é hiperparâmetro
df_normal = pd.read_parquet("../df_normal.parquet")
X_means = df_normal.copy()

Visualização dos clusters com t-SNE

In [ ]:
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt

# Fazer sample dos dados
X_sample = X_means.sample(
    n=20000,          
    random_state=42
)

tsne = TSNE(
    n_components=2,
    random_state=42,
    perplexity=30
)

X_tsne = tsne.fit_transform(X_sample)

plt.figure(figsize=(8, 6))

plt.scatter(
    X_tsne[:, 0],
    X_tsne[:, 1],
    alpha=0.5,
    s=30
)

plt.title('t-SNE antes de K-Means')
plt.xlabel('Componente 1')
plt.ylabel('Componente 2')

plt.tight_layout()
plt.show()

Método do Cotovelo

In [ ]:

from sklearn.cluster import KMeans
import matplotlib.pyplot as plt 
# Lista para guardar a inércia
inertia = []
k_range = range(1, 11) # valores para cluster

for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_means)
    inertia.append(kmeans.inertia_)

# Gráfico do cotovelo
plt.plot(k_range, inertia, marker='o')
plt.xlabel("Número de clusters")
plt.ylabel("Inércia")
plt.title("Método do Cotovelo")
plt.show()

Fit modelo

Métricas

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import (
    silhouette_score,
    davies_bouldin_score,
    calinski_harabasz_score
)

# =========================
# KMEANS
# =========================

kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)

# remove cluster se já existir
X_input = X_means.drop(columns=["cluster"], errors="ignore")

# treinar
X_means["cluster"] = kmeans.fit_predict(X_input)

labels = X_means["cluster"]

# =========================
# MÉTRICAS (CORRETAS)
# =========================

print(f"Silhouette:        {silhouette_score(X_input, labels):.3f}")
print(f"Davies-Bouldin:    {davies_bouldin_score(X_input, labels):.3f}")
print(f"Calinski-Harabasz: {calinski_harabasz_score(X_input, labels):.1f}")
print(f"Inércia:           {kmeans.inertia_:.1f}")

Análise

In [ ]:
centroids = pd.DataFrame(
    kmeans.cluster_centers_,
    columns=X_means.columns[:-1]
)
print(centroids)

# Perfil de cada cluster
X_means.groupby("cluster").mean()

Interpretação

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report

X_k = X_means.drop("cluster", axis=1)
y_k = X_means["cluster"]

tree = DecisionTreeClassifier(
    max_depth=4,
    min_samples_leaf=1000,
    random_state=42
)

# Treinar a árvore com os clusters do KMeans
tree.fit(X_k, y_k)

# Prever os próprios clusters
y_pred = tree.predict(X_k)

# Métricas da árvore
print("\nMétricas da Árvore de Decisão:")
print("Accuracy:", accuracy_score(y_k, y_pred))

print("\nClassification Report:")
print(classification_report(y_k, y_pred))
